# 02 · Build the search index on Kaggle (GPU)

Runs the full offline pipeline (PRD §5.3) on the **real** DeepFashion2 data and produces the artifacts the API loads:

`index.faiss` · `metadata.json` · `manifest.json` · `evaluation.json` · `images/*.webp`

**Kaggle settings:** Accelerator **GPU T4 x1** (or P100) · Internet **on** · Add input
`thusharanair/deepfashion2-original-with-dataframes`. Expected time for 50k items: roughly 30–60 min.
The embedding step is checkpointed in shards, so if the session dies, re-run and it resumes.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/fashion-visual-search.git"   # ← set after you push the repo
if not os.path.exists("fashion-visual-search"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "fashion-visual-search/ml", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "open_clip_torch", "pyarrow"], check=True)
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from pathlib import Path
DATASET = Path("/kaggle/input/deepfashion2-original-with-dataframes")
OUT = Path("/kaggle/working/artifacts")
WORK = Path("/kaggle/working/work")        # embedding checkpoints (resume-safe)
TARGET_ITEMS = 50_000                       # PRD §7.2 working subset

In [ ]:
import logging
from fvs_ml.config import PipelineConfig
from fvs_ml.pipeline import run

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
cfg = PipelineConfig(dataset_root=DATASET, output_dir=OUT, work_dir=WORK, target_items=TARGET_ITEMS,
                     batch_size=512, num_workers=4, seed=42)
report = run(cfg, max_images_per_split=None,
             compat_path=Path("fashion-visual-search/backend/app/data/compatibility.json"), skip_eval=False)

## Results

In [ ]:
import json
ev = json.loads((OUT / "evaluation.json").read_text())
print(json.dumps({k: ev[k] for k in ("catalogue_items", "eval_items", "consumer_to_shop", "complete_the_look", "footprint")}, indent=2))
import pandas as pd
pd.DataFrame(ev["text_to_image"]["per_query"])

## Package for deployment
Images are shipped as **one tar** (tens of thousands of small files upload slowly). Choose one:

* **A — download:** grab `fvs_artifacts.tar` from the Output tab and extract it into `backend/artifacts/` locally.
* **B — push to a private Hugging Face dataset repo** (recommended for Spaces). Add a Kaggle *Secret* named `HF_TOKEN`
  (a write token). The repo **must stay private**: DeepFashion2's terms forbid redistribution.

In [ ]:
import tarfile
with tarfile.open("/kaggle/working/images.tar", "w") as tf:
    tf.add(OUT / "images", arcname="images")
with tarfile.open("/kaggle/working/fvs_artifacts.tar", "w") as tf:
    for name in ("index.faiss", "metadata.json", "manifest.json", "evaluation.json", "images"):
        tf.add(OUT / name, arcname=name)
!ls -lh /kaggle/working/*.tar

In [ ]:
# Option B — upload to a PRIVATE Hugging Face dataset repo
PUSH_TO_HF = False
HF_REPO = "<your-hf-username>/fvs-artifacts"
if PUSH_TO_HF:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    for name in ("index.faiss", "metadata.json", "manifest.json", "evaluation.json"):
        api.upload_file(path_or_fileobj=str(OUT / name), path_in_repo=name, repo_id=HF_REPO, repo_type="dataset")
    api.upload_file(path_or_fileobj="/kaggle/working/images.tar", path_in_repo="images.tar", repo_id=HF_REPO, repo_type="dataset")
    print("uploaded to", HF_REPO)